In [1]:

!pip install PyPDF2 plotly pandas scikit-learn nltk

import re
import PyPDF2
import pandas as pd
import numpy as np
import nltk
import plotly.express as px
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
from nltk.stem import PorterStemmer, WordNetLemmatizer
from sklearn.feature_extraction.text import TfidfVectorizer


nltk.download('punkt')
nltk.download('stopwords')
nltk.download('wordnet')
nltk.download('omw-1.4')

print("Safalta! Saari libraries import ho gayi hain.")


[notice] A new release of pip is available: 26.0.1 -> 26.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip


  Using cached pypdf2-3.0.1-py3-none-any.whl.metadata (6.8 kB)
  Using cached plotly-6.7.0-py3-none-any.whl.metadata (8.6 kB)
  Using cached nltk-3.9.4-py3-none-any.whl.metadata (3.2 kB)
  Using cached narwhals-2.21.2-py3-none-any.whl.metadata (16 kB)
Using cached pypdf2-3.0.1-py3-none-any.whl (232 kB)
Using cached plotly-6.7.0-py3-none-any.whl (9.9 MB)
Using cached nltk-3.9.4-py3-none-any.whl (1.6 MB)
Using cached narwhals-2.21.2-py3-none-any.whl (451 kB)

   ---------------------------------------- 0/5 [regex]
   ---------------------------------------- 0/5 [regex]
   ---------------------------------------- 0/5 [regex]
   ---------------------------------------- 0/5 [regex]
   -------- ------------------------------- 1/5 [PyPDF2]
   -------- ------------------------------- 1/5 [PyPDF2]
   -------- ------------------------------- 1/5 [PyPDF2]
   -------- ------------------------------- 1/5 [PyPDF2]
   -------- ------------------------------- 1/5 [PyPDF2]
   -------- -----------------

[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\Zk\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\Zk\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package wordnet to
[nltk_data]     C:\Users\Zk\AppData\Roaming\nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package omw-1.4 to
[nltk_data]     C:\Users\Zk\AppData\Roaming\nltk_data...


Safalta! Saari libraries import ho gayi hain.


[nltk_data]   Package omw-1.4 is already up-to-date!


In [2]:
import PyPDF2


pdf_path = "advs.pdf"


reader = PyPDF2.PdfReader(pdf_path)


total_pages = len(reader.pages)


raw_text = ""
for page in reader.pages:
    text = page.extract_text()
    if text:
        raw_text += text + " "


print(f"4. Total number of pages in the PDF: {total_pages}")
print("\n5. Sample extracted text from the PDF (First 500 characters):")
print("-" * 50)
print(raw_text[:500])
print("-" * 50)

4. Total number of pages in the PDF: 162

5. Sample extracted text from the PDF (First 500 characters):
--------------------------------------------------
The Adventures of Sherlock Holmes
Arthur Conan Doyle This text is provided to you “as-is” without any warranty. No warranties of any kind, expressed or implied, are made to you as to the text
or any medium it may be on, including but not limited to warranties of merchantablity or ﬁtness for a particular purpose.
This text was formatted from various free ASCII and HTML variants. See http:/ /sherlock-holm.es for an electronic form of this text and
additional information about it.
This text comes f
--------------------------------------------------


In [3]:
print("--- Text Preprocessing Started ---\n")


text_lowered = raw_text.lower()


pattern_numbers = r'\d+'
text_no_numbers = re.sub(pattern_numbers, '', text_lowered)


pattern_symbols = r'[^\w\s]'
text_no_symbols = re.sub(pattern_symbols, '', text_no_numbers)


pattern_spaces = r'\s+'
text_cleaned = re.sub(pattern_spaces, ' ', text_no_symbols).strip()

print(f"Regex Patterns Used:\n- Numbers Pattern: '{pattern_numbers}'\n- Symbols Pattern: '{pattern_symbols}'\n- Extra Spaces Pattern: '{pattern_spaces}'\n")


words = word_tokenize(text_cleaned)


stop_words = set(stopwords.words('english'))
all_stop_words_found = [w for w in words if w in stop_words]
valid_words = [w for w in words if w not in stop_words]


print(f"Total stop words found in the text: {len(all_stop_words_found)}")
print(f"Total valid words after stop word removal: {len(valid_words)}")


sample_valid_words = valid_words[:1000]
ps = PorterStemmer()
stemmed_words = [ps.stem(w) for w in sample_valid_words]


lemmatizer = WordNetLemmatizer()
lemmatized_words = [lemmatizer.lemmatize(w) for w in sample_valid_words]


print("\nSample Stemming Output:", stemmed_words[:10])
print("Sample Lemmatization Output:", lemmatized_words[:10])

--- Text Preprocessing Started ---

Regex Patterns Used:
- Numbers Pattern: '\d+'
- Symbols Pattern: '[^\w\s]'
- Extra Spaces Pattern: '\s+'

Total stop words found in the text: 58028
Total valid words after stop word removal: 47727

Sample Stemming Output: ['adventur', 'sherlock', 'holm', 'arthur', 'conan', 'doyl', 'text', 'provid', 'asi', 'without']
Sample Lemmatization Output: ['adventure', 'sherlock', 'holmes', 'arthur', 'conan', 'doyle', 'text', 'provided', 'asis', 'without']


In [4]:
print("--- Feature Extraction Started ---\n")


unique_words = list(set(sample_valid_words[:50]))
one_hot_data = []

for word in sample_valid_words[:50]:
    row = {u_word: (1 if word == u_word else 0) for u_word in unique_words}
    one_hot_data.append(row)

df_one_hot = pd.DataFrame(one_hot_data)
print("One Hot Encoding Table (Sample 5x5):")
print(df_one_hot.iloc[:5, :5].to_string())


print("\n" + "="*50 + "\n")


chunks = [text_cleaned[i:i+500] for i in range(0, len(text_cleaned[:50000]), 500)]

tfidf = TfidfVectorizer(max_features=50)

tfidf_matrix = tfidf.fit_transform(chunks)

feature_names = tfidf.get_feature_names_out()
print("TF-IDF Feature Names (Total 50):", list(feature_names[:10]), "...\n")


df_tfidf = pd.DataFrame(tfidf_matrix.toarray(), columns=feature_names)
print("TF-IDF Values Table (Sample 5x5):")
print(df_tfidf.iloc[:5, :5].to_string())

--- Feature Extraction Started ---

One Hot Encoding Table (Sample 5x5):
   warranties  expressed  made  version  various
0           0          0     0        0        0
1           0          0     0        0        0
2           0          0     0        0        0
3           0          0     0        0        0
4           0          0     0        0        0


TF-IDF Feature Names (Total 50): ['all', 'an', 'and', 'as', 'at', 'be', 'been', 'but', 'by', 'do'] ...

TF-IDF Values Table (Sample 5x5):
        all        an       and        as   at
0  0.000000  0.141326  0.150189  0.120298  0.0
1  0.000000  0.000000  0.000000  0.000000  0.0
2  0.156020  0.000000  0.212059  0.113237  0.0
3  0.190230  0.000000  0.258557  0.000000  0.0
4  0.160847  0.000000  0.291492  0.116740  0.0


In [5]:
%pip install --upgrade nbformat

mean_tfidf_scores = np.mean(tfidf_matrix.toarray(), axis=0)


df_plot = pd.DataFrame({
    'Words': feature_names,
    'TF-IDF Score': mean_tfidf_scores
}).sort_values(by='TF-IDF Score', ascending=False)


fig = px.scatter(
    df_plot,
    x='Words',
    y='TF-IDF Score',
    size='TF-IDF Score',
    color='TF-IDF Score',
    title='TF-IDF Word Scores Distribution',
    labels={'Words': 'Extracted Feature Words', 'TF-IDF Score': 'Mean TF-IDF Value'}
)


fig.update_layout(xaxis_tickangle=-45)
# fig.show() ki jagah yeh likhein:
fig.show(renderer="notebook_connected")

Note: you may need to restart the kernel to use updated packages.
